## Collection of Category Links

In [4]:
from bs4 import BeautifulSoup
import requests

# load the catalog page
url = 'https://europharma.kz/catalog/zharoponizhayushchiye?segment=available&page=2'
response = requests.get(url)
# Parse HTML
soup = BeautifulSoup(response.text, 'html.parser')

# Extract category links from the catalog menu
links = soup.find_all('a', class_='submenu__link')

# Get category URLs
categories = [link['href'] for link in links]
print(categories)

['/catalog/zharoponizhayushchiye', '/catalog/oporno-dvigatelnaya-sistema', '/catalog/probiotiki-i-prebiotiki', '/catalog/protivovirusnyye-preparaty', '/catalog/protivokashlevyye-preparaty', '/catalog/recepturnye-preparaty', '/catalog/slabitelnye-preparaty', '/catalog/spazmoliticeskie-preparaty', '/catalog/lecenie-zabolevanij-mocepolovoj-sistemy', '/catalog/lecenie-zabolevanij-oporno-dvigatelnogo-apparata', '/catalog/lecheniye-lor-zabolevaniy', '/catalog/lecenie-nevrologiceskih-narusenij', '/catalog/lecenie-peceni', '/catalog/lecenie-pocek', '/catalog/lecenie-ran-i-ozogov', '/catalog/lecenie-saharnogo-diabeta', '/catalog/lecenie-serdecno-sosudistyh-zabolevanij', '/catalog/ginekologiceskie-preparaty', '/catalog/gormonalnye-preparaty', '/catalog/onkologicheskiye-preparaty', '/catalog/analgetiki', '/catalog/ot-parazitov', '/catalog/sredstva-ot-allergii', '/catalog/preparaty-urologii-i-ginekologii', '/catalog/protivoastmaticeskie-preparaty', '/catalog/protivovospalitelnye-sredstva', '/catal

## Collect product links by category

In [7]:
import requests
from bs4 import BeautifulSoup
import time
# base URL of the website
base_url = 'https://europharma.kz'
# URL of the main catalog page
start_url = f'{base_url}/catalog'

# Send a request to the catalog page
response = requests.get(start_url)
# Parse the HTML of the catalog page
soup = BeautifulSoup(response.text, 'html.parser')
# Find all category links in the catalog menu
category_tags = soup.find_all('a', class_='submenu__link')

# Extract unique category URLs
categories = list(set([tag['href'] for tag in category_tags if tag['href'].startswith('/catalog')]))

print(f'Найдено категорий: {len(categories)}')
# List to store all product links
product_links = []

# Iterate through each product category
for category_url in categories:
    print(f'\nКатегория: {category_url}')
    page = 1
     # Iterate through paginated pages within the category
    while True:
        # Build the URL for the current category page
        url = f'{base_url}{category_url}?segment=available&page={page}'
        print(f'  Страница {page}: {url}')
        # Request and parse the category page
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        # Find all anchor tags on the page
        cards = soup.find_all('a', href=True)
        page_links = []
        # Identify product links based on product card images
        for a in cards:
            if a.find('img', class_='card-product__img'):
                full_link = base_url + a['href']
                if full_link not in product_links:
                    page_links.append(full_link)
         # Stop pagination if no products are found
        if not page_links:
            print('  Нет товаров на странице. Переход к следующей категории.')
            break

        product_links.extend(page_links) # Add product links from the current page
        page += 1 # Move to the next page
        time.sleep(1) # Pause to avoid sending too many requests

# Remove duplicate product links
product_links = list(set(product_links))
# Save all collected product links to a text file
with open('products.txt', 'w', encoding='utf-8') as f:
    for link in product_links:
        f.write(link + '\n')
# Print final result
print(f'\nСобрано {len(product_links)} уникальных ссылок на товары. Сохранено в "products.txt".')

Найдено категорий: 69

Категория: /catalog/medicinskie-instrumenty
  Страница 1: https://europharma.kz/catalog/medicinskie-instrumenty?segment=available&page=1
  Страница 2: https://europharma.kz/catalog/medicinskie-instrumenty?segment=available&page=2
  Нет товаров на странице. Переход к следующей категории.

Категория: /catalog/protivogribkovyye-sredstva
  Страница 1: https://europharma.kz/catalog/protivogribkovyye-sredstva?segment=available&page=1
  Страница 2: https://europharma.kz/catalog/protivogribkovyye-sredstva?segment=available&page=2
... (501 lines skipped) ...
  Страница 4: https://europharma.kz/catalog/lecenie-zabolevanij-oporno-dvigatelnogo-apparata?segment=available&page=4
  Страница 5: https://europharma.kz/catalog/lecenie-zabolevanij-oporno-dvigatelnogo-apparata?segment=available&page=5
  Нет товаров на странице. Переход к следующей категории.

Категория: /catalog/bytovaya-tekhnika
  Страница 1: https://europharma.kz/catalog/bytovaya-tekhnika?segment=available&page=1
 

# Product data collection using product links

## Test product data scraper

In [13]:
import requests
from bs4 import BeautifulSoup
# Base URL of the website
base_url = 'https://europharma.kz'

# Category URL (used for testing)
category_url = '/catalog/zharoponizhayushchiye'
page = 1
product_links = []

# Collect product links from the category pages
while True:
    url = f'{base_url}{category_url}?segment=available&page={page}'
    print(f'Парсим страницу {page}: {url}')
    
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    # Find all anchor tags on the page
    cards = soup.find_all('a', href=True)
    page_links = []
    # Identify product links by product card images
    for a in cards:
        if a.find('img', class_='card-product__img'):
            full_link = base_url + a['href']
            if full_link not in product_links:
                page_links.append(full_link)
    # Stop when no products are found
    if not page_links:
        print('Нет товаров на этой странице. Завершаем.')
        break

    product_links.extend(page_links)
    page += 1

# Select only the first product link for testing
first_product_links = product_links[:1]  

# Collect data for the selected product
for link in first_product_links:
    print(f"Собираем данные с: {link}")
    response = requests.get(link)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Dictionary to store product data
    product_data = {}

    try:
        # Product name
        title = soup.select_one("h1.product__title")
        product_data['Название'] = title.text.strip() if title else 'Нет'
    except Exception as e:
        product_data['Название'] = 'Ошибка получения данных'

    try:
        # Цена товара
        price = soup.select_one("span.product__price-value")
        product_data['Цена'] = price.text.strip() if price else 'Нет'
    except Exception as e:
        product_data['Цена'] = 'Ошибка получения данных'

    try:
        # Product price
        installment = soup.select_one("div.bcc-product__price")
        product_data['Рассрочка'] = installment.text.strip() if installment else 'Нет рассрочки'
    except Exception as e:
        product_data['Рассрочка'] = 'Ошибка получения данных'

    try:
        # Installment information
        availability = soup.select_one("dd.characteristic__value.product__available--yes")
        product_data['Наличие'] = availability.text.strip() if availability else 'Нет информации'
    except Exception as e:
        product_data['Наличие'] = 'Ошибка получения данных'

    try:
        # Product model and country
        characteristics = soup.select("dd.characteristic__value")
        if characteristics:
            product_data['Модель'] = characteristics[1].text.strip() if characteristics[1] else 'Нет'
            product_data['Страна'] = characteristics[2].text.strip() if characteristics[2] else 'Нет'
    except Exception as e:
        product_data['Модель'] = 'Ошибка получения данных'

    try:
        # Product model and country
        manufacturer = soup.select_one("a[href^='/manufacturer/']")
        product_data['Производитель'] = manufacturer.text.strip() if manufacturer else 'Нет'
    except Exception as e:
        product_data['Производитель'] = 'Ошибка получения данных'

    try:
        # Product description
        description = soup.select_one("#tab1")
        product_data['Описание'] = description.text.strip() if description else 'Нет'
    except Exception as e:
        product_data['Описание'] = 'Ошибка получения данных'

    try:
        # Product image URL
        image = soup.select_one("div.product-images__wpr img")
        product_data['Изображение'] = image['src'] if image else 'Нет изображения'
    except Exception as e:
        product_data['Изображение'] = 'Ошибка получения данных'

    try:
        # Pharmacies and availability
        pharmacies = soup.select("#tab3 > div.product__pharmacies .product__pharmacy-item")
        pharmacies_list = []
        for pharmacy in pharmacies:
            pharmacy_name = pharmacy.select_one(".product__pharmacy-title").text.strip()
            pharmacy_address = pharmacy.select_one(".product__pharmacy-address").text.strip()
            availability = pharmacy.select_one(".product__pharmacy-badge span")
            availability_status = 'Есть в наличии' if availability and availability['title'] == '9.000' else 'Мало в наличии'
            pharmacies_list.append(f'{pharmacy_name} ({pharmacy_address}) - {availability_status}')
        product_data['Аптеки'] = pharmacies_list if pharmacies_list else 'Нет аптек'
    except Exception as e:
        product_data['Аптеки'] = 'Ошибка получения данных'

    # print collected data for the test product
    print(product_data)

Парсим страницу 1: https://europharma.kz/catalog/zharoponizhayushchiye?segment=available&page=1
Парсим страницу 2: https://europharma.kz/catalog/zharoponizhayushchiye?segment=available&page=2
Парсим страницу 3: https://europharma.kz/catalog/zharoponizhayushchiye?segment=available&page=3
Нет товаров на этой странице. Завершаем.
Собираем данные с: https://europharma.kz/ibukem-tabletki-200-mg-no10
{'Название': 'Ибукем таблетки 200 мг №10', 'Цена': '390', 'Рассрочка': '98 x 4 мес.', 'Наличие': 'Есть в наличии', 'Модель': '8902269269827', 'Страна': 'Индия', 'Производитель': 'Alkem Laboratories Ltd', 'Описание': 'Нет', 'Изображение': 'https://st.europharma.kz/cache/product/49667/400x400_65dc23f3ef8ef.jpg', 'Аптеки': ['Аптека №309 (ул. Тимирязева, дом № 42/30) - Мало в наличии']}


In [14]:
print(product_data)

{'Название': 'Ибукем таблетки 200 мг №10', 'Цена': '390', 'Рассрочка': '98 x 4 мес.', 'Наличие': 'Есть в наличии', 'Модель': '8902269269827', 'Страна': 'Индия', 'Производитель': 'Alkem Laboratories Ltd', 'Описание': 'Нет', 'Изображение': 'https://st.europharma.kz/cache/product/49667/400x400_65dc23f3ef8ef.jpg', 'Аптеки': ['Аптека №309 (ул. Тимирязева, дом № 42/30) - Мало в наличии']}


In [58]:
import csv
import requests
from bs4 import BeautifulSoup

# save product data to a CSV file
def save_to_csv(data, filename="products_data.csv"):
    with open(filename, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=data.keys())
        if file.tell() == 0:  # Write header only if the file is empty
            writer.writeheader()
        writer.writerow(data)

# Read product links from a text file
def read_links_from_file(filename="products.txt"):
    with open(filename, "r", encoding="utf-8") as file:
        return [line.strip() for line in file.readlines()]

# Load product links
product_links = read_links_from_file("products.txt")

# Counter for processed products
product_count = 0

# Iterate over each product link
for link in product_links:
    print(f"Собираем данные с: {link}")
    response = requests.get(link)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Dictionary to store product information
    product_data = {}

    try:
        # Product name
        title = soup.select_one("h1.product__title")
        product_data['Название'] = title.text.strip() if title else 'Нет'
    except Exception as e:
        product_data['Название'] = 'Ошибка получения данных'

    try:
        # Product price
        price = soup.select_one("span.product__price-value")
        product_data['Цена'] = price.text.strip() if price else 'Нет'
    except Exception as e:
        product_data['Цена'] = 'Ошибка получения данных'

    try:
        # Installment information
        installment = soup.select_one("div.bcc-product__price")
        product_data['Рассрочка'] = installment.text.strip() if installment else 'Нет рассрочки'
    except Exception as e:
        product_data['Рассрочка'] = 'Ошибка получения данных'

    try:
        # Product availability
        availability = soup.select_one("dd.characteristic__value.product__available--yes")
        product_data['Наличие'] = availability.text.strip() if availability else 'Нет информации'
    except Exception as e:
        product_data['Наличие'] = 'Ошибка получения данных'

    try:
        # Product model and country
        characteristics = soup.select("dd.characteristic__value")
        if characteristics:
            product_data['Модель'] = characteristics[1].text.strip() if characteristics[1] else 'Нет'
            product_data['Страна'] = characteristics[2].text.strip() if characteristics[2] else 'Нет'
    except Exception as e:
        try:
            # Fallback selector if the main one fails
            model = soup.select_one("body > main > div > div.product > div.product-detail > dl > dd:nth-child(4)")
            product_data['Модель'] = model.text.strip() if model else 'Нет'
        except Exception as e:
            product_data['Модель'] = 'Ошибка получения данных'
            # Fallback selector if the main one fails
            model = soup.select_one("body > main > div > div.product > div.product-detail > dl > dd:nth-child(4)")
            if model:
                print(f"Модель найдена: {model.text.strip()}")
            else:
                print("Модель не найдена по селектору.")

    try:
        # Manufacturer name
        manufacturer = soup.select_one("a[href^='/manufacturer/']")
        product_data['Производитель'] = manufacturer.text.strip() if manufacturer else 'Нет'
    except Exception as e:
        product_data['Производитель'] = 'Ошибка получения данных'

    try:
        # Product description
        description = soup.select_one("#tab1")
        product_data['Описание'] = description.text.strip() if description else 'Нет'
    except Exception as e:
        product_data['Описание'] = 'Ошибка получения данных'

    try:
        # Product image URL
        image = soup.select_one("div.product-images__wpr img")
        product_data['Изображение'] = image['src'] if image else 'Нет изображения'
    except Exception as e:
        product_data['Изображение'] = 'Ошибка получения данных'

    try:
        # Pharmacies where the product is available
        pharmacies = soup.select("#tab3 > div.product__pharmacies .product__pharmacy-item")
        pharmacies_list = []
        for pharmacy in pharmacies:
            pharmacy_name = pharmacy.select_one(".product__pharmacy-title").text.strip()
            pharmacy_address = pharmacy.select_one(".product__pharmacy-address").text.strip()
            availability = pharmacy.select_one("div.product__pharmacy-badge > span")
            pharmacies_list.append(f'{pharmacy_name} ({pharmacy_address}')
        product_data['Аптеки'] = pharmacies_list if pharmacies_list else 'Нет аптек'
    except Exception as e:
        product_data['Аптеки'] = 'Ошибка получения данных'
    try:
    # Product categories (excluding the last breadcrumb)
        categories = soup.select("body > main > div > ul > li:nth-child(n):not(:last-child)")
        categories_list = [category.text.strip() for category in categories]
        product_data['Категории'] = categories_list if categories_list else 'Нет категорий'
    except Exception as e:
        product_data['Категории'] = 'Ошибка получения данных'

    # Save product data to CSV
    save_to_csv(products_data)

    product_count += 1
    if product_count % 20 == 0:
        print(f"Сохранено {product_count} товаров.")

Собираем данные с: https://europharma.kz/dekaris-50-mg-no-2-tabl
Собираем данные с: https://europharma.kz/911-badyaga-100-ml-gel-2
Собираем данные с: https://europharma.kz/analgin-50-rastvor-dinyektsiy-2-ml-no-10-amp
Собираем данные с: https://europharma.kz/askorbinka-vitaminka-sofia-s-aromatom-limona-no10-tabl
Собираем данные с: https://europharma.kz/la-roche-tsikaplast-guby-baryernyy-balzam-75-gr
Собираем данные с: https://europharma.kz/floresan-krem-dlya-zashchity-ot-solntsa-na-sushe-i-na-more-spf-50-africa-kids-150ml
Собираем данные с: https://europharma.kz/pipetka-glaznaya-v-futlyare
Собираем данные с: https://europharma.kz/frutonyanya-pyure-morkov-4mesyatsev-80-g
Собираем данные с: https://europharma.kz/huggies-elite-soft-pants-4-mega-38x2
Собираем данные с: https://europharma.kz/sito-dlya-peska-maloye
... (6544 lines skipped) ...
Собираем данные с: https://europharma.kz/colgate-zubnaa-setka-detskaa-2
Собираем данные с: https://europharma.kz/grelka-kombinirovannaya-tip-b-1-v-ind-

In [4]:
import pandas as pd
df = pd.read_csv('products_data.csv')
print(df["Изображение"].iloc[2])

https://st.europharma.kz/cache/product/5049/400x400_5ee87d2d32256.jpg


In [3]:
print(df['Категории'].isna().sum())

1551


In [67]:
df.shape[0]-1551

4701

### Выравнивание данных

In [7]:
import pandas as pd

# Загружаем данные
df = pd.read_csv('products_data.csv')

# Проходим по всем строкам
for i, row in df.iterrows():
    if pd.isna(row['Категории']) or row['Категории'] == '':
        # Перемещаем данные направо (например, в колонку 'Цена' или любую другую колонку)
        df.at[i, 'Категории'] = df.at[i, 'Аптеки']  # Пример сдвига категории на цену
        df.at[i, 'Аптеки'] = df.at[i, 'Изображение']  # Очищаем колонку 'Цена', если перемещаем данные в неё
        df.at[i, 'Изображение'] = df.at[i, 'Описание'] 
        df.at[i, 'Описание'] = 'Нет'  # Очищаем колонку 'Цена', если перемещаем данные в неё
# Сохраняем результат в новый CSV
df.to_csv('products_data.csv', index=False)
